# Task 6: time until the user's next interaction

Every test row gives a user and the book of their next interaction; the gap since their last training interaction is asked.
Validation rows are built the same way: each user's last training interaction is held out, the target is its gap to the
previous one. The metric is most likely the mean absolute error in seconds, so the models predict the (conditional) median.

- **median baseline**: the median of the user's previous gaps (the assignment's baseline)
- **linear median autoregression**: previous gaps -> next gap, absolute loss (PML Sec. 7.1, Sec. 2.2.2)
- **LightGBM (L1 loss)**: previous gaps + calendar + when the next book appears relative to the user's last interaction

All hyperparameters: `submission/lib/config.py`, Task 6 section.

In [1]:
%load_ext autoreload
%autoreload 2

# assignment1.py and lib/ live in ./submission (this notebook lives in assignment1_2026)
import sys
from pathlib import Path
sys.path.append(str(Path.cwd() / "submission"))

import torch

import assignment1
from lib.config import TASK6_SUBMITTED_GAP_MODEL
from lib.models.gap_regressor import load_gap_model, store_gap_model

print(torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

2.14.1+cu130 | CUDA available: True | NVIDIA GeForce RTX 5080


### Data, target and metric
Each user's interactions in time order with timestamps $t_{u,1} < \dots < t_{u,n}$. The test asks for the gap between $t_{u,n}$ and the user's next
interaction (its book is given). Validation rows: each user's last training interaction is held out,
$$\Delta_u = t_{u,n} - t_{u,n-1}\ \text{(seconds)}, \qquad g_{u,k} = \log\big(1 + t_{u,k} - t_{u,k-1}\big)$$
$$\mathrm{MAE} = \frac{1}{|U|}\sum_u\big|\hat\Delta_u - \Delta_u\big|$$
The best constant under the MAE is the median (PML Ex. 2.4) $\rightarrow$ every model predicts a (conditional) median.

In [2]:
# training file in chronological order -> one held-out gap per user (validation) + the test rows
task6 = assignment1.load_task6()

## Models

### Median baseline (the assignment's baseline)
$$\hat\Delta_u = \operatorname{median}_k\big(t_{u,k} - t_{u,k-1}\big)$$

In [3]:
# baseline: median of the user's previous gaps (seconds)
median_baseline_result = assignment1.train_task6_median_baseline(task6)
store_gap_model(median_baseline_result, "task6", "median_baseline")

[04:11:58] median gap baseline | validation MAE 12,052,136 s (139.5 days)
[04:11:58] stored median_baseline -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task6\median_baseline.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task6/median_baseline.pt')

### Linear median autoregression (PML Sec. 7.1 autoregression + Sec. 2.2.2 MAE)
$$\hat y_u = \theta_0 + \theta_1 g_{u,n-1} + \theta_2 g_{u,n-2} + \theta_3 g_{u,n-3} + \theta_4\operatorname{median}_k g_{u,k} + \theta_5\operatorname{mean}_k g_{u,k} + \theta_6\log(1+n_u)$$
$$\min_{\boldsymbol\theta}\ \sum_u\Big|\log(1+\Delta_u) - \hat y_u\Big|, \qquad \hat\Delta_u = e^{\hat y_u} - 1$$
(features standardised; missing earlier gaps = the user's median gap). The median survives the monotone back-transformation, the mean would not.
Score: 5-fold out-of-fold MAE in seconds.

In [4]:
# linear median autoregression on log(1 + gap) (CPU, a few minutes: one linear program per fold)
linear_gap_result = assignment1.train_task6_linear_gap_model(task6)
store_gap_model(linear_gap_result, "task6", "linear_median_gap_model")

[04:14:50] linear_median gap model | validation MAE 9,885,998 s (114.4 days)  (out-of-fold, 5 folds)
[04:15:44] stored linear_median_gap_model -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task6\linear_median_gap_model.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task6/linear_median_gap_model.pt')

### LightGBM with L1 loss
$$\hat y_u = \sum_{t=1}^{T}\eta\,f_t(\mathbf x_u), \qquad \min\ \sum_u\Big|\log(1+\Delta_u) - \hat y_u\Big|, \qquad \hat\Delta_u = e^{\hat y_u}-1$$
$\mathbf x_u$: the last three gaps $g_{u,n-1}, g_{u,n-2}, g_{u,n-3}$; median / mean / std / min / max of the user's gaps; share of gaps under one day;
$\log(1+n_u)$; active span; year, hour and weekday of $t_{u,n-1}$; and the next book $i$: $\log(1+|U_i|)$, its first and median day minus the user's
last day, and the share of its interactions after that day (a book first read by anyone after $t_{u,n-1}$ bounds the gap from below).
Score: 5-fold out-of-fold MAE in seconds.

In [5]:
# LightGBM with L1 loss on log(1 + gap) (CPU, one to two minutes)
lightgbm_gap_result = assignment1.train_task6_lightgbm_gap_model(task6)
store_gap_model(lightgbm_gap_result, "task6", "lightgbm_gap_model")

[04:15:48] lightgbm gap model | validation MAE 7,225,264 s (83.6 days)  (out-of-fold, 5 folds)
[04:15:49] gap model inputs: last_interaction_year 1923, book_first_day_minus_last_day 1881, book_median_day_minus_last_day 1820, log_book_popularity 1758, last_log_gap 1264, third_last_log_gap 1171, book_share_read_after_last_day 1112, second_last_log_gap 974, std_log_gap 879, median_log_gap 851, log_active_span 785, mean_log_gap 768, max_log_gap 748, min_log_gap 676, log_interactions 536, last_interaction_weekday 351, share_gaps_under_one_day 314, last_interaction_hour 189
[04:15:50] stored lightgbm_gap_model -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task6\lightgbm_gap_model.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task6/lightgbm_gap_model.pt')

### Submission
`delta` = $\hat\Delta_u$ in seconds of the model chosen in `TASK6_SUBMITTED_GAP_MODEL`.

In [6]:
# the submitted model (TASK6_SUBMITTED_GAP_MODEL in lib/config.py) writes submission/output/task_6_predictions.csv
# -> set it to the model with the lowest validation MAE above
submitted_gap_result = load_gap_model("task6", f"{TASK6_SUBMITTED_GAP_MODEL.method}_gap_model")
assignment1.write_task6_predictions(submitted_gap_result.test_predictions)
print("wrote", assignment1.TASK6_PREDICTION_FILE)

wrote C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\submission\output\task_6_predictions.csv
